<a href="https://colab.research.google.com/github/haskinse/bee2041_empirical_project/blob/main/source_code/02_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive # connect to google drive
drive.mount("/content/drive")

project_path = "/content/drive/MyDrive/bee2041_empirical_project" # define project path

Mounted at /content/drive


In [3]:
import pandas as pd # library for data manipulation and tables

In [4]:
def clean_wiki_table(table): # a function to clean the wikipedia tables into structured data frames

  table.columns = table.iloc[0]  # set first row as column names
  table = table.reset_index(drop=True) # reset index cleanly

  table = table.replace(r"\[.*?\]", "", regex = True) # remove footnotes like [1] or [a]

  # extract the release date from the album details column text
  table["release_date"] = table["Album details"].str.extract(r"Released:\s*([A-Za-z]+\s+\d{1,2},\s+\d{4})")

  table["release_date"] = pd.to_datetime(table['release_date']) # convert the date from string to datetime format for easier sorting

  table["label"] = table["Album details"].str.extract(r"Label:\s*(.*?)\s*Formats:") # extract the label from the details column and store in a new column

  table["us_sales"] = table["Sales[a]"].str.extract(r"US:\s*([\d,]+)") # extract the us sales
  table["uk_sales"] = table["Sales[a]"].str.extract(r"UK:\s*([\d,]+)") # extract the uk sales

  # for the sales data, remove commas, convert to numeric and fill missing values with zero
  table[["uk_sales", "us_sales"]] = (table[["uk_sales", "us_sales"]].replace(",", "", regex=True).apply(pd.to_numeric, errors="coerce").fillna(0).astype(int))

  table["riaa"] = table["Certifications"].str.extract(r"RIAA:\s*(.*?)(?:ARIA:|BPI:|$)") # extract the riaa certification text
  riaa_mult = pd.to_numeric(table["riaa"].str.extract(r"(\d+)")[0], errors="coerce").fillna(1).astype(int) # extract the riaa mulripliers like 2x

  # convert the riaa certification levels into estimated units
  table["riaa_units"] = (table["riaa"].str.contains("Gold", na = False) * 500_000 + table['riaa'].str.contains("Platinum", na=False) * 1_000_000) * riaa_mult

  # extract bpi certification text
  table["bpi"] = table["Certifications"].str.extract(r"BPI:\s*(.*?)(?:RIAA:|ARIA:|MC:|RMNZ:|IFPI DEN:|IFPI NOR:|GLF:|$)")

  bpi_mult = pd.to_numeric(table["bpi"].str.extract(r"(\d+)")[0], errors="coerce").fillna(1).astype(int) # extract bpi multipliers

  # convert the bpi certification levels into estimated units
  table["bpi_units"] = (table["bpi"].str.contains("Gold", na = False) * 100_000 + table["bpi"].str.contains("Platinum", na=False) * 300_000) * bpi_mult

  # remove columns that won't be used any further
  table = table.drop(columns = ["Album details", "Certifications", "AUS [6]", "CAN [8]", "DEN [16]", "IRE [17]", "NZ [9]", "NOR [18]", "SPA [13]", "SWE [14]", "Sales[a]"])

  table = table.rename(columns = {"Title": "album_name", "US [19]": "us_peak", "UK [10]": "uk_peak"}) # standardise column names
  table = table.dropna(subset=["release_date"]).reset_index(drop=True) # drop non-album rows where there is no release date
  table = table.dropna(axis = 1, how = "all") # drop empty columns

  return table # return the cleaned data frame

In [5]:
wiki_studio_albums = pd.read_csv(f"{project_path}/data/raw/wiki_studio_albums.csv") # read raw csv data into a data frame
wiki_studio_albums = clean_wiki_table(wiki_studio_albums) # clean the albums table using the function

wiki_studio_albums.head() # show the first five rows

,album_name,us_peak,uk_peak,release_date,label,us_sales,uk_sales,riaa,riaa_units,bpi,bpi_units
0,Taylor Swift,5,81,2006-10-24,Big Machine,5871000,251320,8× Platinum,8000000,Platinum,300000
1,Fearless,1,5,2008-11-11,Big Machine,7286000,718518,11× Platinum (Diamond),11000000,2× Platinum,600000
2,Speak Now,1,6,2010-10-25,Big Machine,4817000,416676,6× Platinum,6000000,Platinum,300000
3,Red,1,1,2012-10-22,Big Machine,4582000,874532,8× Platinum,8000000,3× Platinum,900000
4,1989,1,1,2014-10-27,Big Machine,6472000,1792380,14× Platinum (Diamond),14000000,6× Platinum,1800000


In [6]:
wiki_tv_albums = pd.read_csv(f"{project_path}/data/raw/wiki_tv_albums.csv") # read raw csv data into a data frame
wiki_tv_albums = clean_wiki_table(wiki_tv_albums) # clean the albums table using the function

wiki_tv_albums.head() # show the first five rows

,album_name,us_peak,uk_peak,release_date,label,us_sales,uk_sales,riaa,riaa_units,bpi,bpi_units
0,Fearless (Taylor's Version),1,1,2021-04-09,Republic,737000,217947,4× Platinum,4000000,Platinum,300000
1,Red (Taylor's Version),1,1,2021-11-12,Republic,950000,357565,6× Platinum,6000000,Platinum,300000
2,Speak Now (Taylor's Version),1,1,2023-07-07,Republic,908000,209302,4× Platinum,4000000,Platinum,300000
3,1989 (Taylor's Version),1,1,2023-10-27,Republic,2389000,390226,4× Platinum,4000000,2× Platinum,600000


In [7]:
album_data = pd.concat([wiki_studio_albums, wiki_tv_albums], ignore_index = True) # combine studio_albums and rerecorded_albums
album_data = album_data.sort_values("release_date") # sort by release date
album_data = album_data.reset_index(drop = True) # reset the index
album_data["album_id"] = album_data.index + 1 # create the album id column to uniquely identify each album

album_data.to_csv(f"{project_path}/data/clean/album_data.csv", index = False) # save clean data to drive

album_data.head() # show first five rows

,album_name,us_peak,uk_peak,release_date,label,us_sales,uk_sales,riaa,riaa_units,bpi,bpi_units,album_id
0,Taylor Swift,5,81,2006-10-24,Big Machine,5871000,251320,8× Platinum,8000000,Platinum,300000,1
1,Fearless,1,5,2008-11-11,Big Machine,7286000,718518,11× Platinum (Diamond),11000000,2× Platinum,600000,2
2,Speak Now,1,6,2010-10-25,Big Machine,4817000,416676,6× Platinum,6000000,Platinum,300000,3
3,Red,1,1,2012-10-22,Big Machine,4582000,874532,8× Platinum,8000000,3× Platinum,900000,4
4,1989,1,1,2014-10-27,Big Machine,6472000,1792380,14× Platinum (Diamond),14000000,6× Platinum,1800000,5


In [8]:
import re # library for regular expressions

In [9]:
def clean_name(s): # a function to clean names

  if pd.isna(s): # if the name is nan
    return s # return nan

  s = s.lower().strip() # convert to lowercase and get rid of whitespace

  # standardise quotation marks and apostophes
  s = s.replace("’", "'").replace("‘", "'")
  s = s.replace("“", '"').replace("”", '"')

  # replace symbols with text equivalents
  s = s.replace("&", "and")
  s = s.replace("+", "and")

  # remove elipses
  s = s.replace("...", " ")

  # remove content in brackets like (ft)
  s = re.sub(r"\(.*?\)", "", s)
  s = re.sub(r"\[.*?\]", "", s)

  s = re.sub(r"[^\w\s']", "", s) # remove punctuation while keeping letters, spaces and apostrophes

  s = re.sub(r"\s+", " ", s) # remove extra spaces

  return s.strip() # return clean track name

In [10]:
# map the taylor's versions to the original album names used in the Kaggle dataset
tv_to_original = {
  "Fearless (Taylor's Version)": "Fearless Platinum Edition",
  "Red (Taylor's Version)": "Red (Deluxe Edition)",
  "Speak Now (Taylor's Version)": "Speak Now (Deluxe Edition)",
  "1989 (Taylor's Version) [Deluxe]": "1989 (Deluxe Edition)"
}

In [11]:
# map alternative names onto standardised ids
album_map = {
  "Taylor Swift": "1",
  "Fearless Platinum Edition": "2",
  "Speak Now (Deluxe Edition)": "3",
  "Red (Deluxe Edition)": "4",
  "1989 (Deluxe Edition)": "5",
  "reputation": "6",
  "Lover": "7",
  "folklore (deluxe version)": "8",
  "evermore (deluxe version)": "9",
  "Fearless (Taylor's Version)": "10",
  "Red (Taylor's Version)": "11",
  "Midnights (The Til Dawn Edition)": "12",
  "Speak Now (Taylor's Version)": "13",
  "1989 (Taylor's Version) [Deluxe]": "14",
  "THE TORTURED POETS DEPARTMENT: THE ANTHOLOGY": "15",
  "The Life of a Showgirl": "16",
}

In [12]:
kaggle_data = pd.read_csv(f"{project_path}/data/raw/kaggle_data.csv") # read raw csv data into a data frame

kaggle_data = kaggle_data.loc[:, ~kaggle_data.columns.str.contains("^Unnamed")] # remove extra unnamed column
kaggle_data["track_name"] = kaggle_data["track_name"].apply(clean_name) # standardise the track names
kaggle_data["full_key"] = kaggle_data["key"] + " " + kaggle_data["mode"] # combine key and mode into one column
kaggle_data = kaggle_data.drop(columns = ["release_date", "album_id", "key", "mode"]) # drop unnecessary columns

tv_rows = kaggle_data[kaggle_data["album_name"].isin(tv_to_original.keys())].copy() # identify rows corresponding to taylor's version rows
tv_rows["album_name"] = tv_rows["album_name"].replace(tv_to_original) # replaces album names with their original names
kaggle_data = pd.concat([kaggle_data, tv_rows], ignore_index = True) # adds the new rows back to dataset, so it contains both tv and normal albums
kaggle_data["album_name"] = kaggle_data["album_name"].map(album_map).fillna("Other") # map album names to standardised album ids
kaggle_data = kaggle_data.rename(columns = {"album_name": "album_id"}) # renames album_name to album_id

kaggle_data.head() # show first five rows

,track_name,album_id,tempo,energy,danceability,duration_min,time_signature,valence,acousticness,instrumentalness,liveness,loudness,speechiness,full_key
0,fortnight,15,192.004,38.6,50.4,3.816083,4/4,28.1,50.20,0.00153,9.61,-10.976,3.08,B Major
1,the tortured poets department,15,110.259,42.8,60.4,4.884133,4/4,29.2,4.83,0.00000,12.60,-8.441,2.55,C Major
2,my boy only breaks his favorite toys,15,97.073,56.3,59.6,3.396683,4/4,48.1,13.70,0.00000,30.20,-7.362,2.69,C Major
3,down bad,15,159.707,36.6,54.1,4.353800,4/4,16.8,56.00,0.00010,9.46,-10.412,7.48,B Major
4,so long london,15,160.218,53.3,42.3,4.382917,4/4,24.8,73.00,0.26400,8.16,-11.388,32.20,A Major


In [13]:
spotify_data = pd.read_csv(f"{project_path}/data/raw/spotify_data.csv") # read raw csv data into a data frame

spotify_data["track_name"] = spotify_data["track_name"].apply(clean_name) # clean track names using the function previously used on the kaggle_data
spotify_data["album_name"] = spotify_data["album_name"].map(album_map).fillna("Other") # map album names to standardised album ids
spotify_data = spotify_data[spotify_data["album_name"] != "Other"] # get rid of tracks in "other"
spotify_data = spotify_data.rename(columns = {"album_name": "album_id"}) # rename album_name column to album_id

spotify_data.head() # show first five rows

,artist_name,album_id,release_date,track_name,track_number,explicit
19,Taylor Swift,16,2025-10-03,the fate of ophelia,1,False
20,Taylor Swift,16,2025-10-03,elizabeth taylor,2,False
21,Taylor Swift,16,2025-10-03,opalite,3,False
22,Taylor Swift,16,2025-10-03,father figure,4,True
23,Taylor Swift,16,2025-10-03,eldest daughter,5,True


In [14]:
track_data = spotify_data.merge(kaggle_data, on = ["album_id", "track_name"], how = "inner") # combine kaggle and spotify data into one dataframe
track_data = track_data.sort_values(["release_date", "track_number"]) # sort by release date
track_data = track_data.reset_index(drop = True) # reset the index
track_data["track_id"] = track_data.index + 1 # create track_id column

track_data.to_csv(f"{project_path}/data/clean/track_data.csv", index = False) # save clean data to drive

track_data.head() # show first five rows

,artist_name,album_id,release_date,track_name,track_number,explicit,tempo,energy,danceability,duration_min,time_signature,valence,acousticness,instrumentalness,liveness,loudness,speechiness,full_key,track_id
0,Taylor Swift,1,2006-10-24,tim mcgraw,1,False,76.009,49.1,58.0,3.86845,4/4,42.5,57.5,0.0,12.10,-6.462,2.51,C Major,1
1,Taylor Swift,1,2006-10-24,picture to burn,2,False,105.586,87.7,65.8,2.88445,4/4,82.1,17.3,0.0,9.62,-2.098,3.23,G Major,2
2,Taylor Swift,1,2006-10-24,teardrops on my guitar radio single remix,3,False,99.953,41.7,62.1,3.38400,4/4,28.9,28.8,0.0,11.90,-6.941,2.31,Bb Major,3
3,Taylor Swift,1,2006-10-24,a place in this world,4,False,115.028,77.7,57.6,3.32000,4/4,42.8,5.1,0.0,32.00,-2.881,3.24,A Major,4
4,Taylor Swift,1,2006-10-24,cold as you,5,False,175.558,48.2,41.8,3.98355,4/4,26.1,21.7,0.0,12.30,-5.769,2.66,F Major,5


In [15]:
lastfm_data = pd.read_csv(f"{project_path}/data/raw/lastfm_data.csv") # read raw csv data into a data frame

# data is already in a consistent clean format

artist_data = lastfm_data # rename data frame

artist_data.to_csv(f"{project_path}/data/clean/artist_data.csv", index = False) # save clean data to drive

artist_data.head() # show first five rows

,artist_name,listeners,playcount
0,Bon Iver,3549046,237752276
1,Brendon Urie,20539,303651
2,Chris Stapleton,872359,30187834
3,Colbie Caillat,2050317,36757055
4,Ed Sheeran,4189377,254582074
